In [1]:
import os
import glob
import pandas as pd

In [13]:
df = pd.read_excel(r"D:\Ricci\Экспозиция\new_history_03082026.xlsx")

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 95253 entries, 0 to 141429
Data columns (total 48 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   batch                     95253 non-null  int64         
 1   uid                       95253 non-null  object        
 2   ID Корпуса                95253 non-null  int64         
 3   ID ЖК                     95253 non-null  int64         
 4   ЖК рус                    95253 non-null  object        
 5   ЖК англ                   39800 non-null  object        
 6   Корпус                    95253 non-null  object        
 7   кр Корпус                 95253 non-null  object        
 8   Регион                    95253 non-null  object        
 9   ID кв                     95253 non-null  int64         
 10  Дата актуализации         95253 non-null  datetime64[ns]
 11  Комнат                    95135 non-null  float64       
 12  Площадь               

In [69]:
columns = ['ID Корпуса', 'ID ЖК', 'ЖК рус', 'кр Корпус', 'Регион', 'Дата актуализации', 'Комнат', 'Площадь', 'Этаж', 'Номер секции', 'Адрес корп', 'lat', 'lng', 'Район Город', 'Округ Направление', 'АТД', 'Тип кв/ап', 'Застройщик', 'Тип помещения', 'Договор К', 'Сдача К', 'Стадия К', 'Цена со скидкой', 'Зона', 'Отделка текст', 'Старт продаж К', 'ID дом.рф', 'Класс Ricci']
print(len(columns))

28


In [27]:
exclude = [
    'Нежилое',
    'Кладовка',
    'Машино-место',
    'Офис',
    'Таунхаус(дом)'
]

df = df[~df['Тип помещения'].isin(exclude)].copy()

In [28]:
df['Тип помещения'].unique()

array(['Квартира', 'Апартамент', 'Кв/ап'], dtype=object)

In [17]:
df['Дата актуализации'] = pd.to_datetime(
    df['Дата актуализации'],
    format='%d.%m.%Y'
)

In [19]:
df['Дата актуализации'] = df['Дата актуализации'].dt.to_period('M').dt.to_timestamp()

In [20]:
df['Дата актуализации'].unique()

<DatetimeArray>
['2026-08-01 00:00:00']
Length: 1, dtype: datetime64[ns]

In [21]:
df['Регион'].unique()

array(['Московская область', 'Москва', 'Новая Москва'], dtype=object)

In [24]:
import json

with open(r'C:\PycharmProjects\ndv_parcing\!haracteristik_dictionary\projects.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

id_to_name = {}

for name, info in data.items():
    try:
        id_to_name[int(info['id'])] = name
    except (TypeError, ValueError):
        continue

# Добавляем столбец в датафрейм
df['Название ЖК НДВ'] = df['ID ЖК'].map(id_to_name)

In [35]:
with open(r'C:\PycharmProjects\ndv_parcing\area_dictionary\output.json', 'r', encoding='utf-8') as f:
    area_json = json.load(f)


def enrich_area_typology(df, area_json):
    df = df.copy()

    df['Площадь'] = (
        df['Площадь']
        .astype(str)
        .str.replace(',', '.')
        .str.replace(' ', '')
        .astype(float)
    )

    developers_to_skip = {'А101', 'Аквилон'}
    jk_name_to_skip = {'Гармония Парк', 'Мишино-2'}
    jk_name_to_skip2 = {'Серебро', 'Берег'}

    for idx, row in df.iterrows():
        jk_name = str(row['Название ЖК НДВ']).strip()
        area = row['Площадь']
        developer = str(row['Застройщик']).strip()

        if pd.isna(jk_name) or pd.isna(area):
            df.at[idx, 'Комнат НДВ'] = 'Н/Д'
            continue

        if area <= 28:
            df.at[idx, 'Комнат НДВ'] = 'студия'
            continue

        # if developer in developers_to_skip or jk_name in jk_name_to_skip:
        #     continue

        found = False

        if jk_name in area_json:
            jk_dict = area_json[jk_name]
            area = round(area, 2)

            for json_area_str, room_type in jk_dict.items():
                try:
                    if area == round(float(json_area_str), 2):
                        df.at[idx, 'Комнат НДВ'] = (
                            'студия' if str(room_type).lower() in ['0', 'st', 'ст'] else room_type
                        )
                        found = True
                        break
                except ValueError:
                    pass

            if not found:
                candidates = []

                for json_area_str, room_type in jk_dict.items():
                    try:
                        json_area = round(float(json_area_str), 2)
                        if abs(area - json_area) <= 3:
                            candidates.append((abs(area - json_area), room_type))
                    except ValueError:
                        pass

                if candidates:
                    _, closest_room = min(candidates, key=lambda x: x[0])
                    df.at[idx, 'Комнат НДВ'] = (
                        'студия' if str(closest_room).lower() in ['0', 'st', 'ст'] else closest_room
                    )
                    found = True

    return df

result = enrich_area_typology(df, area_json)

In [74]:
result.info()

<class 'pandas.core.frame.DataFrame'>
Index: 93594 entries, 0 to 141429
Data columns (total 50 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   batch                     93594 non-null  int64         
 1   uid                       93594 non-null  object        
 2   ID Корпуса                93594 non-null  int64         
 3   ID ЖК                     93594 non-null  int64         
 4   ЖК рус                    93594 non-null  object        
 5   ЖК англ                   39800 non-null  object        
 6   Корпус                    93594 non-null  object        
 7   кр Корпус                 93594 non-null  object        
 8   Регион                    93594 non-null  object        
 9   ID кв                     93594 non-null  int64         
 10  Дата актуализации         93594 non-null  datetime64[ns]
 11  Комнат                    93581 non-null  object        
 12  Площадь               

In [79]:
result['Стадия К'] = (
    result['Стадия К']
    .str.replace('_', ' ', regex=False)
    .str.capitalize()
)

In [38]:
result['Комнат НДВ'] = result['Комнат НДВ'].fillna(result['Комнат'])

In [42]:
result['Комнат'] = result['Комнат НДВ']

In [80]:
result['Стадия К'].value_counts()

Стадия К
Нижние этажи     27465
Идёт отделка     21715
Сдан гк          18384
Котлован         14837
Верхние этажи    11193
Name: count, dtype: int64

In [48]:
result['Тип помещения'] = result['Тип помещения'].replace('Кв/ап', 'Квартира')

In [50]:
unique_values = (
    result['Отделка текст']
    .dropna()          # убрать пропуски (если нужно)
    .drop_duplicates() # оставить только уникальные значения
    .sort_values()     # необязательно: отсортировать
    .to_frame(name='Отделка текст')
)

unique_values.to_excel('unique_otdelka.xlsx', index=False)

In [55]:
result['Отделка текст'] = result['Отделка текст'].fillna(result['Отделка помещения'])

In [59]:
result['Отделка текст'].unique()

array(['С отделкой', 'White Box', 'Без отделки'], dtype=object)

In [58]:
result['Отделка текст'] = result['Отделка текст'].replace({
    '0 - Нет': 'Без отделки',
    '1 - Есть': 'С отделкой',
    '3 - Неизвестно': 'White Box'
})

In [52]:
# Загружаем файл со справочником
mapping_df = pd.read_excel(r'C:\PycharmProjects\ndv_parcing\ricci\unique_otdelka.xlsx')

# Создаем словарь
mapping = dict(zip(mapping_df['Отделка текст'], mapping_df['Отделка new']))

# Заменяем значения
result['Отделка текст'] = result['Отделка текст'].replace(mapping)

In [53]:
result['Отделка текст'].unique()

array(['С отделкой', 'White Box', nan, 'Без отделки'], dtype=object)

In [61]:
# Загружаем справочник
classes = pd.read_excel(r'C:\PycharmProjects\ndv_parcing\ricci\06072026 классы (1).xlsx')

# Создаем словарь: ЖК рус -> Класс Ricci
mapping = classes.set_index('ЖК рус')['Класс Ricci'].to_dict()

# Добавляем столбец в result
result['Класс Ricci'] = result['ЖК рус'].map(mapping)

In [65]:
missing = (
    result.loc[result['Класс Ricci'].isna(), ['ЖК рус']]
          .drop_duplicates()
          .sort_values('ЖК рус')
)

missing.to_excel('missing_ricci_class.xlsx', index=False)

In [64]:
result = result[result['Застройщик'] != 'Московский фонд реновации'].copy()

In [71]:
result['Цена со скидкой'] = result['Цена со скидкой'].fillna(result['Цена'])

In [73]:
result = result[
    result['Цена со скидкой'].notna() &
    (result['Цена со скидкой'] != 0)
].copy()

In [75]:
columns = [
    'ID Корпуса', 'ID ЖК', 'ЖК рус', 'кр Корпус', 'Регион',
    'Дата актуализации', 'Комнат', 'Площадь', 'Этаж',
    'Номер секции', 'Адрес корп', 'lat', 'lng',
    'Район Город', 'Округ Направление', 'АТД',
    'Тип кв/ап', 'Застройщик', 'Тип помещения',
    'Договор К', 'Сдача К', 'Стадия К',
    'Цена со скидкой', 'Зона', 'Отделка текст',
    'Старт продаж К', 'ID дом.рф', 'Класс Ricci'
]

result_new = result[columns].copy()

In [77]:
result_new.to_csv('exposition0826.csv', index=False)

In [76]:
result_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 93594 entries, 0 to 141429
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ID Корпуса         93594 non-null  int64         
 1   ID ЖК              93594 non-null  int64         
 2   ЖК рус             93594 non-null  object        
 3   кр Корпус          93594 non-null  object        
 4   Регион             93594 non-null  object        
 5   Дата актуализации  93594 non-null  datetime64[ns]
 6   Комнат             93581 non-null  object        
 7   Площадь            93594 non-null  float64       
 8   Этаж               93586 non-null  float64       
 9   Номер секции       66003 non-null  object        
 10  Адрес корп         93594 non-null  object        
 11  lat                93594 non-null  float64       
 12  lng                93594 non-null  float64       
 13  Район Город        93594 non-null  object        
 14  Округ Напр